In [1]:
!python -m pip install -q pandas numpy scikit-learn



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
%%writefile code.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, Literal, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)

READMITTED_CLASSES_3: Tuple[str, str, str] = ("NO", "<30", ">30")


class TargetEngineeringError(ValueError):
    """Raised when the readmitted target contains unexpected/invalid values."""


def _normalize_readmitted_value(v: object) -> Optional[str]:
    """
    Normalize a single readmitted value:
    - None/NaN -> None
    - strings -> stripped, uppercased (except <30 and >30 remain as-is after upper)
    """
    if v is None:
        return None
    # pandas NA / numpy nan
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass

    s = str(v).strip()
    if s == "":
        return None
    return s.upper()


def validate_readmitted_values(
    values: Union[pd.Series, Iterable[object]],
    allowed: Sequence[str] = READMITTED_CLASSES_3,
    *,
    allow_na: bool = False,
) -> None:
    """
    Validate that all non-missing readmitted values are inside 'allowed'.

    Raises:
        TargetEngineeringError if an unexpected value is found.
    """
    allowed_set = {a.upper() for a in allowed}
    if isinstance(values, pd.Series):
        raw_iter = values.tolist()
    else:
        raw_iter = list(values)

    unexpected = set()
    for v in raw_iter:
        nv = _normalize_readmitted_value(v)
        if nv is None:
            if not allow_na:
                # Missing values are unexpected unless allow_na=True
                unexpected.add(None)
            continue
        if nv not in allowed_set:
            unexpected.add(nv)

    if unexpected:
        raise TargetEngineeringError(
            f"Unexpected readmitted values found: {sorted([x for x in unexpected if x is not None])}"
            + (" (and missing values)" if None in unexpected else "")
            + f". Allowed values: {list(allowed)}."
        )


def binarize_readmitted(
    readmitted: pd.Series,
    *,
    positive_value: str = "<30",
    negative_values: Sequence[str] = ("NO", ">30"),
    output_name: str = "readmitted_30d",
    unknown_policy: Literal["error", "nan"] = "error",
) -> pd.Series:
    """
    Convert {NO, <30, >30} into binary {1 if <30, 0 otherwise}.

    Args:
        readmitted: pandas Series with original readmitted values.
        positive_value: value mapped to 1.
        negative_values: values mapped to 0.
        output_name: name for the output series.
        unknown_policy:
            - "error": raise if any unexpected/NA values appear
            - "nan": map unexpected/NA to <NA> (nullable integer)

    Returns:
        pd.Series of dtype int8 (or nullable Int8 if unknown_policy="nan").
    """
    pos = positive_value.upper()
    neg = tuple(v.upper() for v in negative_values)

    # Normalize series
    norm = readmitted.map(_normalize_readmitted_value)

    mapping: Dict[str, int] = {pos: 1, **{v: 0 for v in neg}}

    if unknown_policy == "error":
        validate_readmitted_values(norm, allowed=(pos, *neg), allow_na=False)
        out = norm.map(mapping).astype(np.int8)
        out.name = output_name
        return out

    if unknown_policy == "nan":
        # allow NA/unknown -> <NA>
        out = norm.map(mapping)
        out = out.astype("Int8")  # nullable integer supports <NA>
        out.name = output_name
        return out

    raise ValueError(f"unknown_policy must be 'error' or 'nan', got: {unknown_policy}")


def keep_multiclass_readmitted(
    readmitted: pd.Series,
    *,
    output_name: str = "readmitted_3class",
    allowed: Sequence[str] = READMITTED_CLASSES_3,
    unknown_policy: Literal["error", "nan"] = "error",
) -> pd.Series:
    """
    Keep the original 3-class label, optionally validating values.

    Returns:
        pd.Series of dtype 'category' with categories in the allowed order,
        or with missing if unknown_policy="nan".
    """
    norm = readmitted.map(_normalize_readmitted_value)

    if unknown_policy == "error":
        validate_readmitted_values(norm, allowed=allowed, allow_na=False)
    elif unknown_policy == "nan":
        # allow NA/unknown; do not raise
        pass
    else:
        raise ValueError(f"unknown_policy must be 'error' or 'nan', got: {unknown_policy}")

    cat = pd.Categorical(norm, categories=[a.upper() for a in allowed], ordered=False)
    out = pd.Series(cat, index=readmitted.index, name=output_name)
    return out


def engineer_targets(
    df: pd.DataFrame,
    *,
    source_col: str = "readmitted",
    binary_col: str = "readmitted_30d",
    multiclass_col: str = "readmitted_3class",
    add_multiclass: bool = True,
    drop_source: bool = False,
    unknown_policy: Literal["error", "nan"] = "error",
) -> pd.DataFrame:
    """
    Add engineered target columns to a dataframe:
      - binary: 1 if <30 else 0
      - optional multiclass: categorical {NO, <30, >30}

    This keeps Section 2 self-contained and reproducible.

    Returns:
        A copy of df with new columns added (and optionally source_col dropped).
    """
    if source_col not in df.columns:
        raise KeyError(f"Column '{source_col}' not found in df. Available: {list(df.columns)}")

    out = df.copy()
    out[binary_col] = binarize_readmitted(
        out[source_col],
        output_name=binary_col,
        unknown_policy=unknown_policy,
    )

    if add_multiclass:
        out[multiclass_col] = keep_multiclass_readmitted(
            out[source_col],
            output_name=multiclass_col,
            unknown_policy=unknown_policy,
        )

    if drop_source:
        out.drop(columns=[source_col], inplace=True)

    return out


@dataclass(frozen=True)
class BinaryConfusionTerms:
    tp: int
    fp: int
    fn: int
    tn: int


def confusion_terms_binary(
    y_true: Union[pd.Series, np.ndarray, Sequence[int]],
    y_pred: Union[pd.Series, np.ndarray, Sequence[int]],
    *,
    positive_label: int = 1,
) -> BinaryConfusionTerms:
    """
    Return TP/FP/FN/TN for a binary task once the positive class is defined.

    This directly supports the Section 2 narrative about TP/FP/FN/TN meaning.
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    # cm layout with labels [0,1] is:
    # [[TN, FP],
    #  [FN, TP]]
    tn, fp, fn, tp = cm.ravel()
    return BinaryConfusionTerms(tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn))


def metrics_binary(
    y_true: Union[pd.Series, np.ndarray, Sequence[int]],
    y_pred: Union[pd.Series, np.ndarray, Sequence[int]],
    y_score: Optional[Union[pd.Series, np.ndarray, Sequence[float]]] = None,
) -> Dict[str, Optional[float]]:
    """
    Convenience metric bundle for binary framing (Section 2 awareness):
      - accuracy
      - f1
      - balanced_accuracy
      - roc_auc (if y_score is provided)
    """
    res: Dict[str, Optional[float]] = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, pos_label=1)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "roc_auc": None,
    }
    if y_score is not None:
        res["roc_auc"] = float(roc_auc_score(y_true, y_score))
    return res


def metrics_multiclass(
    y_true: Union[pd.Series, np.ndarray, Sequence[str]],
    y_pred: Union[pd.Series, np.ndarray, Sequence[str]],
    y_proba: Optional[np.ndarray] = None,
    *,
    labels: Sequence[str] = READMITTED_CLASSES_3,
) -> Dict[str, Optional[float]]:
    """
    Metric bundle for the advanced extension (3-class framing):
      - macro_f1
      - weighted_f1
      - balanced_accuracy (macro recall)
      - ovr_auc_macro (if y_proba is provided with shape [n_samples, n_classes])

    Notes:
      - For AUC, we use one-vs-rest multi-class AUC (macro).
      - Requires y_proba columns correspond to 'labels' in the same order.
    """
    labels_up = [l.upper() for l in labels]

    yt = pd.Series(y_true).map(_normalize_readmitted_value)
    yp = pd.Series(y_pred).map(_normalize_readmitted_value)

    res: Dict[str, Optional[float]] = {
        "macro_f1": float(f1_score(yt, yp, labels=labels_up, average="macro")),
        "weighted_f1": float(f1_score(yt, yp, labels=labels_up, average="weighted")),
        "balanced_accuracy": float(balanced_accuracy_score(yt, yp)),
        "ovr_auc_macro": None,
    }

    if y_proba is not None:
        y_proba = np.asarray(y_proba)
        if y_proba.ndim != 2 or y_proba.shape[1] != len(labels_up):
            raise ValueError(
                f"y_proba must have shape [n_samples, {len(labels_up)}] matching labels order {labels_up}."
            )
        res["ovr_auc_macro"] = float(
            roc_auc_score(yt, y_proba, multi_class="ovr", average="macro", labels=labels_up)
        )

    return res


Overwriting code.py


In [14]:
%%writefile test_target_engineering.py
import os
import unittest
import importlib.util

import numpy as np
import pandas as pd

# Robust import of /content/.../code.py without colliding with the stdlib 'code' module
def import_module_from_path(module_name: str, file_path: str):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot import module from {file_path}")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


PROJECT_DIR = os.path.dirname(os.path.abspath(__file__))
CODE_PATH = os.path.join(PROJECT_DIR, "code.py")
mod = import_module_from_path("project_code", CODE_PATH)


class TestTargetEngineering(unittest.TestCase):
    def test_binarize_basic_mapping(self):
        s = pd.Series(["NO", "<30", ">30", "NO", ">30", "<30"])
        y = mod.binarize_readmitted(s, unknown_policy="error")
        self.assertEqual(list(y.astype(int)), [0, 1, 0, 0, 0, 1])
        self.assertEqual(y.name, "readmitted_30d")
        self.assertTrue(str(y.dtype).lower() in ("int8", "int64", "int32"))

    def test_binarize_handles_whitespace_and_case(self):
        s = pd.Series([" no ", " <30", ">30 ", "No", "<30", " >30"])
        y = mod.binarize_readmitted(s, unknown_policy="error")
        self.assertEqual(list(y.astype(int)), [0, 1, 0, 0, 1, 0])

    def test_binarize_unknown_raises(self):
        s = pd.Series(["NO", "MAYBE", "<30"])
        with self.assertRaises(mod.TargetEngineeringError):
            _ = mod.binarize_readmitted(s, unknown_policy="error")

    def test_binarize_unknown_to_nan(self):
        s = pd.Series(["NO", "MAYBE", "<30", None])
        y = mod.binarize_readmitted(s, unknown_policy="nan")
        # Expect: NO->0, MAYBE->NA, <30->1, None->NA
        self.assertEqual(int(y.iloc[0]), 0)
        self.assertTrue(pd.isna(y.iloc[1]))
        self.assertEqual(int(y.iloc[2]), 1)
        self.assertTrue(pd.isna(y.iloc[3]))
        self.assertEqual(str(y.dtype), "Int8")

    def test_engineer_targets_adds_columns(self):
        df = pd.DataFrame(
            {
                "readmitted": ["NO", "<30", ">30"],
                "age": ["[50-60)", "[60-70)", "[40-50)"],
            }
        )
        out = mod.engineer_targets(df, add_multiclass=True, drop_source=False, unknown_policy="error")
        self.assertIn("readmitted_30d", out.columns)
        self.assertIn("readmitted_3class", out.columns)
        self.assertIn("readmitted", out.columns)
        self.assertEqual(list(out["readmitted_30d"].astype(int)), [0, 1, 0])
        # multiclass should be categorical with expected categories
        self.assertTrue(pd.api.types.is_categorical_dtype(out["readmitted_3class"]))

    def test_confusion_terms_binary(self):
        y_true = [1, 1, 0, 0, 1, 0]
        y_pred = [1, 0, 0, 1, 1, 0]
        terms = mod.confusion_terms_binary(y_true, y_pred)
        # Manually:
        # TP: positions 0 and 4 => 2
        # FN: position 1 => 1
        # FP: position 3 => 1
        # TN: positions 2 and 5 => 2
        self.assertEqual((terms.tp, terms.fn, terms.fp, terms.tn), (2, 1, 1, 2))

    def test_metrics_multiclass_with_auc(self):
        y_true = ["NO", "<30", ">30", "NO", "<30", ">30"]
        y_pred = ["NO", "<30", "NO", "NO", "<30", ">30"]
        # Probabilities aligned with labels order ("NO", "<30", ">30")
        y_proba = np.array(
            [
                [0.8, 0.1, 0.1],
                [0.1, 0.8, 0.1],
                [0.4, 0.2, 0.4],
                [0.7, 0.2, 0.1],
                [0.1, 0.7, 0.2],
                [0.2, 0.1, 0.7],
            ],
            dtype=float,
        )
        res = mod.metrics_multiclass(y_true, y_pred, y_proba=y_proba)
        for key in ["macro_f1", "weighted_f1", "balanced_accuracy", "ovr_auc_macro"]:
            self.assertIn(key, res)
            self.assertIsInstance(res[key], float)

    def test_keep_multiclass_error_policy(self):
        s = pd.Series(["NO", "<30", "???"])
        with self.assertRaises(mod.TargetEngineeringError):
            _ = mod.keep_multiclass_readmitted(s, unknown_policy="error")

    def test_keep_multiclass_nan_policy(self):
        s = pd.Series(["NO", "<30", "???"])
        out = mod.keep_multiclass_readmitted(s, unknown_policy="nan")
        self.assertTrue(pd.isna(out.iloc[2]))


if __name__ == "__main__":
    unittest.main(verbosity=2)


Overwriting test_target_engineering.py


In [15]:
!python -m unittest -v test_target_engineering.py


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Python313\Lib\unittest\__main__.py", line 18, in <module>
    main(module=None)
    ~~~~^^^^^^^^^^^^^
  File "c:\Python313\Lib\unittest\main.py", line 103, in __init__
    self.parseArgs(argv)
    ~~~~~~~~~~~~~~^^^^^^
  File "c:\Python313\Lib\unittest\main.py", line 142, in parseArgs
    self.createTests()
    ~~~~~~~~~~~~~~~~^^
  File "c:\Python313\Lib\unittest\main.py", line 153, in createTests
    self.test = self.testLoader.loadTestsFromNames(self.testNames,
                ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
                                                   self.module)
                                                   ^^^^^^^^^^^^
  File "c:\Python313\Lib\unittest\loader.py", line 207, in loadTestsFromNames
    suites = [self.loadTestsFromName(name, module) for name in names]
              ~~~~~~~~~~~~~~~~~~~~~~^^^

In [2]:
%%writefile section3_dataset_understanding.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd


DEFAULT_MISSING_TOKENS: Tuple[str, ...] = (
    "", "?", "NA", "N/A", "NONE", "NULL", "NAN", "UNKNOWN", "UNKNOWN/INVALID"
)

DEFAULT_ID_COLUMNS: Tuple[str, ...] = (
    "id",
    "url",
    "encounter_id",
    "patient_nbr",
    "patient_id",
    "paper_id",
)

DEFAULT_TEXT_COLUMNS: Tuple[str, ...] = (
    "title",
    "abstract",
    "text",
)

DEFAULT_INTERVAL_COLUMNS: Tuple[str, ...] = (
    "year",
)


def _is_missing_value(x: Any, *, treat_empty_as_missing: bool = True) -> bool:
    if x is None:
        return True
    try:
        if pd.isna(x):
            return True
    except Exception:
        pass
    if treat_empty_as_missing and isinstance(x, str) and x.strip() == "":
        return True
    return False


def _normalize_token(s: str) -> str:
    return s.strip().upper()


def _count_missing_like_tokens(
    series: pd.Series,
    *,
    missing_tokens: Sequence[str] = DEFAULT_MISSING_TOKENS,
) -> Dict[str, int]:
    """
    Count occurrences of missing/unknown encodings in an object/string-like column.
    Counting is case-insensitive and strips whitespace.
    """
    if not (pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)):
        return {}

    toks = {_normalize_token(t) for t in missing_tokens}
    counts: Dict[str, int] = {t: 0 for t in toks}

    for v in series.astype("object").tolist():
        if v is None:
            continue
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass

        if isinstance(v, str):
            key = _normalize_token(v)
        else:
            key = _normalize_token(str(v))

        if key in counts:
            counts[key] += 1

    # Keep only tokens that actually appear (>0)
    return {k: v for k, v in counts.items() if v > 0}


def _unique_ratio(series: pd.Series) -> float:
    n = len(series)
    if n == 0:
        return 0.0
    return float(series.nunique(dropna=True)) / float(n)


def _avg_text_length(series: pd.Series) -> float:
    vals = []
    for v in series.tolist():
        if _is_missing_value(v, treat_empty_as_missing=True):
            continue
        vals.append(len(str(v)))
    if not vals:
        return 0.0
    return float(np.mean(vals))


def infer_kind_and_scale(
    series: pd.Series,
    col: str,
    *,
    text_columns: Sequence[str] = DEFAULT_TEXT_COLUMNS,
    interval_columns: Sequence[str] = DEFAULT_INTERVAL_COLUMNS,
    ordinal_columns: Sequence[str] = (),
) -> Tuple[str, str]:
    """
    Returns (kind, scale) where:
      - kind  in {"numeric","categorical","text"}
      - scale in {"nominal","ordinal","interval","ratio","text"}
    """
    col_low = col.strip().lower()

    # Forced text columns by name (common in this project)
    if col_low in {c.lower() for c in text_columns}:
        return "text", "text"

    # Numeric
    if pd.api.types.is_numeric_dtype(series.dtype) and not pd.api.types.is_bool_dtype(series.dtype):
        if col_low in {c.lower() for c in interval_columns}:
            return "numeric", "interval"
        return "numeric", "ratio"

    # Bool
    if pd.api.types.is_bool_dtype(series.dtype):
        return "categorical", "nominal"

    # Otherwise object/string => decide text vs categorical by heuristic
    avg_len = _avg_text_length(series)
    if avg_len >= 30.0:
        return "text", "text"

    # Ordinal hint by name override
    if col_low in {c.lower() for c in ordinal_columns}:
        return "categorical", "ordinal"

    return "categorical", "nominal"


def infer_role(
    df: pd.DataFrame,
    col: str,
    kind: str,
    *,
    id_columns: Sequence[str] = DEFAULT_ID_COLUMNS,
    derived_text_column: str = "text",
    id_like_unique_ratio_threshold: float = 0.98,
) -> str:
    """
    Returns a role label to help the report/checklist:
      - "identifier"
      - "raw_text_feature"
      - "derived_feature"
      - "metadata_feature"
      - "numeric_feature"
      - "categorical_feature"
    """
    col_low = col.strip().lower()
    s = df[col]

    if col_low in {c.lower() for c in id_columns}:
        return "identifier"

    # Name-based derived text (common: text = title + abstract)
    if col_low == derived_text_column.lower():
        return "derived_feature"

    # ID-like heuristic (very high uniqueness ratio)
    if _unique_ratio(s) >= id_like_unique_ratio_threshold and kind != "numeric":
        return "identifier"

    # Heuristic: metadata-like
    if col_low in {"year", "venue"}:
        return "metadata_feature"

    if kind == "text":
        return "raw_text_feature"
    if kind == "numeric":
        return "numeric_feature"
    return "categorical_feature"


@dataclass(frozen=True)
class DatasetUnderstandingReport:
    data_dictionary: pd.DataFrame
    missing_summary: pd.DataFrame
    missing_tokens_found: pd.DataFrame
    duplicates_summary: pd.DataFrame
    leakage_risks: pd.DataFrame
    quality_risks: pd.DataFrame


def build_data_dictionary(
    df: pd.DataFrame,
    *,
    text_columns: Sequence[str] = DEFAULT_TEXT_COLUMNS,
    interval_columns: Sequence[str] = DEFAULT_INTERVAL_COLUMNS,
    ordinal_columns: Sequence[str] = (),
    id_columns: Sequence[str] = DEFAULT_ID_COLUMNS,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    n = len(df)

    for col in df.columns:
        s = df[col]
        kind, scale = infer_kind_and_scale(
            s,
            col,
            text_columns=text_columns,
            interval_columns=interval_columns,
            ordinal_columns=ordinal_columns,
        )
        role = infer_role(df, col, kind, id_columns=id_columns)

        miss = int(s.isna().sum())
        empty = 0
        if pd.api.types.is_object_dtype(s.dtype) or pd.api.types.is_string_dtype(s.dtype):
            empty = int((s.astype("object").map(lambda x: isinstance(x, str) and x.strip() == "")).sum())

        nun = int(s.nunique(dropna=True))
        ur = float(nun) / float(n) if n else 0.0

        # Example values (up to 5) excluding missing/empty
        examples = []
        for v in s.tolist():
            if _is_missing_value(v, treat_empty_as_missing=True):
                continue
            examples.append(v)
            if len(examples) >= 5:
                break

        rows.append(
            {
                "column": col,
                "pandas_dtype": str(s.dtype),
                "kind": kind,               # numeric / categorical / text
                "scale": scale,             # nominal / ordinal / interval / ratio / text
                "role": role,               # identifier / feature etc
                "n_rows": n,
                "n_unique": nun,
                "unique_ratio": ur,
                "missing_count": miss,
                "missing_rate": (miss / n) if n else 0.0,
                "empty_string_count": empty,
                "examples": examples,
            }
        )

    return pd.DataFrame(rows).sort_values(["role", "kind", "column"]).reset_index(drop=True)


def summarize_missingness(
    df: pd.DataFrame,
    *,
    treat_empty_as_missing: bool = True,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    n = len(df)

    for col in df.columns:
        s = df[col]
        na = int(s.isna().sum())
        empty = 0
        if treat_empty_as_missing and (pd.api.types.is_object_dtype(s.dtype) or pd.api.types.is_string_dtype(s.dtype)):
            empty = int((s.astype("object").map(lambda x: isinstance(x, str) and x.strip() == "")).sum())

        miss_total = na + empty
        rows.append(
            {
                "column": col,
                "na_count": na,
                "empty_string_count": empty,
                "missing_total": miss_total,
                "missing_rate": (miss_total / n) if n else 0.0,
            }
        )

    return pd.DataFrame(rows).sort_values("missing_rate", ascending=False).reset_index(drop=True)


def detect_missing_tokens(
    df: pd.DataFrame,
    *,
    missing_tokens: Sequence[str] = DEFAULT_MISSING_TOKENS,
) -> pd.DataFrame:
    """
    Detects tokens like '?', 'N/A', 'Unknown', etc., per column (string/object columns).
    Returns a long-form DataFrame: column, token, count
    """
    rows: List[Dict[str, Any]] = []
    for col in df.columns:
        series = df[col]
        counts = _count_missing_like_tokens(series, missing_tokens=missing_tokens)
        for token, cnt in sorted(counts.items()):
            rows.append({"column": col, "token": token, "count": int(cnt)})
    return pd.DataFrame(rows)


def duplicates_report(
    df: pd.DataFrame,
    *,
    key_columns: Sequence[str] = ("url",),
    content_columns: Sequence[str] = ("title", "abstract"),
) -> pd.DataFrame:
    """
    Returns a compact table of duplicate signals:
      - duplicates by key_columns (e.g., URL duplicates)
      - duplicates by content_columns (title+abstract duplicates)
    """
    rows: List[Dict[str, Any]] = []

    def _dup_stats(subset: Sequence[str], name: str) -> None:
        present = [c for c in subset if c in df.columns]
        if not present:
            rows.append(
                {
                    "scope": name,
                    "subset": list(subset),
                    "available_subset": [],
                    "duplicate_rows": 0,
                    "duplicate_groups": 0,
                }
            )
            return

        dup_mask = df.duplicated(subset=present, keep=False)
        duplicate_rows = int(dup_mask.sum())
        # number of duplicated keys/groups (excluding non-duplicated)
        duplicate_groups = int(df.loc[dup_mask, present].drop_duplicates().shape[0])

        rows.append(
            {
                "scope": name,
                "subset": list(subset),
                "available_subset": present,
                "duplicate_rows": duplicate_rows,
                "duplicate_groups": duplicate_groups,
            }
        )

    _dup_stats(key_columns, "key_duplicates")
    _dup_stats(content_columns, "content_duplicates")

    return pd.DataFrame(rows)


def leakage_risk_report(
    df: pd.DataFrame,
    *,
    id_columns: Sequence[str] = DEFAULT_ID_COLUMNS,
    id_like_unique_ratio_threshold: float = 0.98,
) -> pd.DataFrame:
    """
    Flags columns that are likely to cause leakage / trivial memorization in supervised setups
    (or trivial retrieval shortcuts in IR): identifier-like columns, near-unique columns, etc.
    """
    risks: List[Dict[str, Any]] = []
    n = len(df)

    for col in df.columns:
        s = df[col]
        col_low = col.strip().lower()
        ur = _unique_ratio(s)
        nun = int(s.nunique(dropna=True))

        # 1) Name-based ID columns
        if col_low in {c.lower() for c in id_columns}:
            risks.append(
                {
                    "column": col,
                    "risk_type": "identifier_like",
                    "severity": "high",
                    "reason": "Column name matches a known identifier field (e.g., 'url'/'id').",
                    "unique_ratio": ur,
                    "n_unique": nun,
                    "n_rows": n,
                }
            )
            continue

        # 2) ID-like by uniqueness ratio (very close to 1)
        if ur >= id_like_unique_ratio_threshold and not pd.api.types.is_numeric_dtype(s.dtype):
            risks.append(
                {
                    "column": col,
                    "risk_type": "identifier_like",
                    "severity": "medium",
                    "reason": f"Very high uniqueness ratio (>= {id_like_unique_ratio_threshold}).",
                    "unique_ratio": ur,
                    "n_unique": nun,
                    "n_rows": n,
                }
            )

    return pd.DataFrame(risks).sort_values(["severity", "unique_ratio"], ascending=[True, False]).reset_index(drop=True)


def quality_risk_report(
    df: pd.DataFrame,
    *,
    missing_rate_warn: float = 0.20,
    high_cardinality_warn: float = 0.50,
    treat_empty_as_missing: bool = True,
) -> pd.DataFrame:
    """
    Heuristic quality risks: high missingness, high cardinality, constant columns.
    """
    risks: List[Dict[str, Any]] = []
    n = len(df)

    for col in df.columns:
        s = df[col]

        # Missingness risk
        miss_rate = float(
            (_is_missing_value_count(s, treat_empty_as_missing=treat_empty_as_missing) / n) if n else 0.0
        )
        if miss_rate >= missing_rate_warn:
            risks.append(
                {
                    "column": col,
                    "risk_type": "high_missingness",
                    "severity": "medium",
                    "metric": "missing_rate",
                    "value": miss_rate,
                    "threshold": missing_rate_warn,
                    "note": "Consider explicit handling in preprocessing (drop/impute/tokenize missing).",
                }
            )

        # Cardinality risk (mostly for categorical/text columns)
        ur = float(s.nunique(dropna=True) / n) if n else 0.0
        if ur >= high_cardinality_warn and not pd.api.types.is_numeric_dtype(s.dtype):
            risks.append(
                {
                    "column": col,
                    "risk_type": "high_cardinality",
                    "severity": "low",
                    "metric": "unique_ratio",
                    "value": ur,
                    "threshold": high_cardinality_warn,
                    "note": "May lead to sparse features / overfitting if one-hot encoded.",
                }
            )

        # Constant column risk
        nun = int(s.nunique(dropna=True))
        if nun <= 1:
            risks.append(
                {
                    "column": col,
                    "risk_type": "constant_or_almost_constant",
                    "severity": "low",
                    "metric": "n_unique",
                    "value": nun,
                    "threshold": 1,
                    "note": "Usually safe to drop (no predictive signal).",
                }
            )

    return pd.DataFrame(risks).sort_values(["severity", "risk_type", "column"]).reset_index(drop=True)


def _is_missing_value_count(series: pd.Series, *, treat_empty_as_missing: bool) -> int:
    na = int(series.isna().sum())
    if not treat_empty_as_missing:
        return na
    if pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype):
        empty = int((series.astype("object").map(lambda x: isinstance(x, str) and x.strip() == "")).sum())
        return na + empty
    return na


def profile_dataset(
    df_or_records: Union[pd.DataFrame, Sequence[Dict[str, Any]]],
    *,
    expected_keys: Sequence[str] = ("title", "abstract", "url", "venue", "year"),
    text_columns: Sequence[str] = DEFAULT_TEXT_COLUMNS,
    interval_columns: Sequence[str] = DEFAULT_INTERVAL_COLUMNS,
    ordinal_columns: Sequence[str] = (),
    id_columns: Sequence[str] = DEFAULT_ID_COLUMNS,
    missing_tokens: Sequence[str] = DEFAULT_MISSING_TOKENS,
) -> DatasetUnderstandingReport:
    """
    End-to-end helper for Section 3:
      - Builds data dictionary (types/scales/roles)
      - Summarizes missingness + detects missing tokens
      - Detects duplicates
      - Flags leakage risks
      - Flags quality risks

    Accepts either a DataFrame or the raw list[dict] records.
    """
    if isinstance(df_or_records, pd.DataFrame):
        df = df_or_records.copy()
    else:
        # records -> dataframe
        if not isinstance(df_or_records, (list, tuple)):
            raise TypeError(f"df_or_records must be a DataFrame or list/tuple of dicts, got {type(df_or_records)}")
        rows = []
        for i, rec in enumerate(df_or_records):
            if not isinstance(rec, dict):
                raise TypeError(f"Each record must be a dict. Found {type(rec)} at index {i}.")
            rows.append({k: rec.get(k, None) for k in expected_keys})
        df = pd.DataFrame(rows)

    data_dict = build_data_dictionary(
        df,
        text_columns=text_columns,
        interval_columns=interval_columns,
        ordinal_columns=ordinal_columns,
        id_columns=id_columns,
    )
    missing_summary = summarize_missingness(df, treat_empty_as_missing=True)
    missing_tokens_found = detect_missing_tokens(df, missing_tokens=missing_tokens)
    duplicates_summary = duplicates_report(df)
    leakage_risks = leakage_risk_report(df, id_columns=id_columns)
    quality_risks = quality_risk_report(df)

    return DatasetUnderstandingReport(
        data_dictionary=data_dict,
        missing_summary=missing_summary,
        missing_tokens_found=missing_tokens_found,
        duplicates_summary=duplicates_summary,
        leakage_risks=leakage_risks,
        quality_risks=quality_risks,
    )


Writing section3_dataset_understanding.py


In [3]:
%%writefile test_section3_dataset_understanding.py

import unittest
import pandas as pd

import section3_dataset_understanding as s3


class TestSection3DatasetUnderstanding(unittest.TestCase):
    def _tiny_df(self):
        df = pd.DataFrame(
            {
                "title": ["Paper A", "Paper B", "Paper B"],
                "abstract": ["Text A", "Text B", "Text B"],
                "url": ["u1", "u2", "u2"],   # duplicate to test duplicate detection
                "venue": ["EMNLP", "EMNLP", "?"],  # token-like missing/unknown
                "year": [2016, 2017, 2017],
            }
        )
        df["text"] = df["title"] + " " + df["abstract"]  # derived
        return df

    def test_build_data_dictionary_infers_expected_roles_and_scales(self):
        df = self._tiny_df()
        dd = s3.build_data_dictionary(df)

        cols = set(dd["column"].tolist())
        self.assertTrue({"title", "abstract", "url", "venue", "year", "text"}.issubset(cols))

        year_row = dd.loc[dd["column"] == "year"].iloc[0]
        self.assertEqual(year_row["kind"], "numeric")
        self.assertEqual(year_row["scale"], "interval")  # year should be interval by default
        self.assertIn(year_row["role"], {"metadata_feature", "numeric_feature"})

        url_row = dd.loc[dd["column"] == "url"].iloc[0]
        self.assertEqual(url_row["role"], "identifier")

        text_row = dd.loc[dd["column"] == "text"].iloc[0]
        self.assertEqual(text_row["kind"], "text")
        self.assertEqual(text_row["role"], "derived_feature")

        title_row = dd.loc[dd["column"] == "title"].iloc[0]
        self.assertEqual(title_row["kind"], "text")
        self.assertEqual(title_row["role"], "raw_text_feature")

    def test_detect_missing_tokens_finds_question_mark(self):
        df = self._tiny_df()
        mt = s3.detect_missing_tokens(df, missing_tokens=("?", "N/A"))

        # We expect venue has "?" exactly once
        found = mt[(mt["column"] == "venue") & (mt["token"] == "?")]
        self.assertEqual(int(found["count"].iloc[0]), 1)

    def test_missing_summary_counts_empty_strings(self):
        df = self._tiny_df()
        df.loc[1, "abstract"] = "   "  # empty after strip
        ms = s3.summarize_missingness(df, treat_empty_as_missing=True)

        abs_row = ms.loc[ms["column"] == "abstract"].iloc[0]
        self.assertGreaterEqual(int(abs_row["empty_string_count"]), 1)
        self.assertGreaterEqual(int(abs_row["missing_total"]), 1)

    def test_duplicates_report_detects_key_duplicates(self):
        df = self._tiny_df()
        rep = s3.duplicates_report(df, key_columns=("url",), content_columns=("title", "abstract"))

        key_row = rep.loc[rep["scope"] == "key_duplicates"].iloc[0]
        self.assertEqual(key_row["available_subset"], ["url"])
        self.assertGreaterEqual(int(key_row["duplicate_rows"]), 2)  # u2 repeated => two rows duplicated

    def test_leakage_risk_report_flags_url_as_identifier_like(self):
        df = self._tiny_df()
        lr = s3.leakage_risk_report(df, id_columns=("url", "id"))

        self.assertTrue("column" in lr.columns)
        self.assertTrue(any(lr["column"] == "url"))

    def test_profile_dataset_accepts_records_list(self):
        records = [
            {"title": "T1", "abstract": "A1", "url": "u1", "venue": "EMNLP", "year": 2016},
            {"title": "T2", "abstract": "A2", "url": "u2", "venue": "EMNLP", "year": 2017},
        ]
        report = s3.profile_dataset(records)
        self.assertTrue(hasattr(report, "data_dictionary"))
        self.assertTrue(hasattr(report, "missing_summary"))
        self.assertTrue(hasattr(report, "duplicates_summary"))


if __name__ == "__main__":
    unittest.main(verbosity=2)


Writing test_section3_dataset_understanding.py


In [4]:
!python -m unittest -v test_section3_dataset_understanding.py


test_build_data_dictionary_infers_expected_roles_and_scales (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.test_build_data_dictionary_infers_expected_roles_and_scales) ... ok
test_detect_missing_tokens_finds_question_mark (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.test_detect_missing_tokens_finds_question_mark) ... ok
test_duplicates_report_detects_key_duplicates (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.test_duplicates_report_detects_key_duplicates) ... ok
test_leakage_risk_report_flags_url_as_identifier_like (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.test_leakage_risk_report_flags_url_as_identifier_like) ... ok
test_missing_summary_counts_empty_strings (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.test_missing_summary_counts_empty_strings) ... ok
test_profile_dataset_accepts_records_list (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.

In [5]:
!python -m pip install -q pandas numpy scikit-learn matplotlib



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
%%writefile section4_eda_imbalance.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn import metrics
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split


# ---------------------------------------------------------------------
# 4.1  Class imbalance after binarization
# ---------------------------------------------------------------------


@dataclass(frozen=True)
class ClassImbalanceSummary:
    """Compact summary of class imbalance for the binary target.

    The positive class is the early readmission (<30 days), i.e. y=1.
    This supports the discussion of why accuracy can be misleading and
    why precision/recall/F1 (for the minority class) are preferred
    under imbalance (see PDF: confusion matrix + metrics). 
    """
    N: int
    n_pos: int
    n_neg: int
    pos_rate: float
    neg_rate: float

    def as_dict(self) -> Dict[str, float]:
        return {
            "N": self.N,
            "n_pos": self.n_pos,
            "n_neg": self.n_neg,
            "pos_rate": self.pos_rate,
            "neg_rate": self.neg_rate,
        }


def compute_class_imbalance(
    y: Iterable[int],
    *,
    positive_label: int = 1,
    negative_label: int = 0,
) -> ClassImbalanceSummary:
    """Quantify class imbalance for a binary target y={0,1}.

    Parameters
    ----------
    y:
        Iterable with the binary labels (after binarization).
    positive_label:
        Label considered "positive" (early readmission).
    negative_label:
        Label considered "negative".

    Returns
    -------
    ClassImbalanceSummary
        N, n_pos, n_neg and prevalence rates.
    """
    y_series = pd.Series(list(y))
    N = int(len(y_series))
    counts = y_series.value_counts(dropna=False).to_dict()
    n_pos = int(counts.get(positive_label, 0))
    n_neg = int(counts.get(negative_label, 0))
    pos_rate = float(n_pos / N) if N else 0.0
    neg_rate = float(n_neg / N) if N else 0.0
    return ClassImbalanceSummary(N=N, n_pos=n_pos, n_neg=n_neg,
                                 pos_rate=pos_rate, neg_rate=neg_rate)


# ---------------------------------------------------------------------
# 4.2  Missingness per feature and per record
# ---------------------------------------------------------------------

MISSING_TOKENS: Tuple[str, ...] = (
    "", "?", "NA", "N/A", "NONE", "NULL", "NAN", "UNKNOWN", "UNKNOWN/INVALID"
)


def is_missing(x: Any, missing_tokens: Sequence[str] = MISSING_TOKENS) -> bool:
    """Unified predicate for 'missing' used in Section 4.

    Treats as missing:
    - None / NaN (according to pandas.isna)
    - empty strings (after strip)
    - tokens like '?', 'NA', 'UNKNOWN', 'UNKNOWN/INVALID', etc.
    """
    if x is None:
        return True
    try:
        if pd.isna(x):
            return True
    except Exception:
        # If pandas cannot decide, ignore and continue
        pass

    if isinstance(x, str):
        s = x.strip()
        if s == "":
            return True
        s_up = s.upper()
        tokens_up = {t.upper() for t in missing_tokens}
        return s_up in tokens_up

    return False


def _series_missing_mask(series: pd.Series,
                         missing_tokens: Sequence[str]) -> pd.Series:
    """Return a boolean mask indicating missing values in a Series."""
    return series.map(lambda v: is_missing(v, missing_tokens=missing_tokens))


def missingness_by_feature(
    df: pd.DataFrame,
    *,
    feature_cols: Optional[Sequence[str]] = None,
    missing_tokens: Sequence[str] = MISSING_TOKENS,
) -> pd.DataFrame:
    """Column-level missingness summary.

    Parameters
    ----------
    df:
        Full dataframe including target and features.
    feature_cols:
        Columns to treat as features. If None, all columns except common
        target columns are used.
    missing_tokens:
        Encodings to be treated as missing (in addition to NaN / empty).

    Returns
    -------
    DataFrame
        Index = column name, columns:
        - missing_rate
        - missing_count
    """
    if feature_cols is None:
        excluded = {"y", "readmitted", "readmitted_30d", "readmitted_3class"}
        feature_cols = [c for c in df.columns if c not in excluded]

    n = len(df)
    if n == 0:
        raise ValueError("DataFrame is empty; cannot summarize missingness.")

    def _col_missing_rate(col: pd.Series) -> float:
        return float(_series_missing_mask(col, missing_tokens).mean())

    missing_rate_by_col = (
        df[feature_cols]
        .apply(_col_missing_rate)
        .sort_values(ascending=False)
        .to_frame("missing_rate")
    )
    missing_rate_by_col["missing_count"] = (
        (missing_rate_by_col["missing_rate"] * n).round().astype(int)
    )
    return missing_rate_by_col


@dataclass(frozen=True)
class MissingnessByRowResult:
    """Result of row-level missingness analysis."""
    missing_per_row: pd.Series
    missing_rate_per_row: pd.Series
    summary_counts: Dict[str, float]
    high_missing_fraction: float

    def as_dict(self) -> Dict[str, Any]:
        return {
            "missing_per_row_summary": self.summary_counts,
            "high_missing_fraction": self.high_missing_fraction,
        }


def missingness_by_row(
    df: pd.DataFrame,
    *,
    feature_cols: Optional[Sequence[str]] = None,
    missing_tokens: Sequence[str] = MISSING_TOKENS,
    high_missing_threshold: float = 0.30,
) -> MissingnessByRowResult:
    """Row-level missingness (per encounter/patient record).

    Parameters
    ----------
    df:
        DataFrame with at least the feature columns.
    feature_cols:
        Subset of df columns to treat as features. If None, all except target.
    missing_tokens:
        Tokens to treat as missing.
    high_missing_threshold:
        Threshold on missing *rate* to consider a row 'highly incomplete'.

    Returns
    -------
    MissingnessByRowResult
        - missing_per_row: number of missing values in each row.
        - missing_rate_per_row: fraction of missing per row.
        - summary_counts: describe() of missing_per_row (as dict).
        - high_missing_fraction: proportion of rows with missing_rate >= threshold.
    """
    if feature_cols is None:
        excluded = {"y", "readmitted", "readmitted_30d", "readmitted_3class"}
        feature_cols = [c for c in df.columns if c not in excluded]

    if not feature_cols:
        raise ValueError("No feature columns provided for missingness_by_row.")

    miss_matrix = df[feature_cols].applymap(
        lambda v: is_missing(v, missing_tokens=missing_tokens)
    )
    missing_per_row = miss_matrix.sum(axis=1)
    missing_rate_per_row = missing_per_row / float(len(feature_cols))

    summary_counts = missing_per_row.describe(
        percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
    ).to_dict()
    high_missing_fraction = float((missing_rate_per_row >= high_missing_threshold).mean())

    return MissingnessByRowResult(
        missing_per_row=missing_per_row,
        missing_rate_per_row=missing_rate_per_row,
        summary_counts=summary_counts,
        high_missing_fraction=high_missing_fraction,
    )


# ---------------------------------------------------------------------
# 4.3  Distributions and high-cardinality categoricals
# ---------------------------------------------------------------------


def topk_table(
    series: pd.Series,
    *,
    k: int = 20,
    missing_tokens: Sequence[str] = MISSING_TOKENS,
    missing_label: str = "__MISSING__",
) -> Tuple[pd.DataFrame, int, float]:
    """Frequency table for high-cardinality categorical variables.

    Parameters
    ----------
    series:
        Categorical column (e.g., diag_1, medical_specialty, payer_code).
    k:
        Top-k categories to display.
    missing_tokens:
        Encodings considered missing.
    missing_label:
        Label used to group all missing values into a single bucket.

    Returns
    -------
    (top_df, n_unique, coverage)
        - top_df: DataFrame with columns ['count', 'rate'] for top-k items.
        - n_unique: total number of distinct categories (including missing_label).
        - coverage: fraction of rows covered by the top-k categories.
    """
    s = series.copy()
    s = s.where(~s.map(lambda v: is_missing(v, missing_tokens=missing_tokens)),
                other=missing_label)

    vc = s.value_counts(dropna=False)
    top = vc.head(k).to_frame("count")
    if len(s) > 0:
        top["rate"] = top["count"] / float(len(s))
        coverage = float(top["count"].sum() / float(len(s)))
    else:
        top["rate"] = 0.0
        coverage = 0.0

    return top, int(vc.shape[0]), coverage


# ---------------------------------------------------------------------
# 4.4  Clinically meaningful errors (FN vs FP) – dummy baseline
# ---------------------------------------------------------------------


@dataclass(frozen=True)
class DummyBaselineResult:
    """Result of a most_frequent DummyClassifier baseline.

    This baseline is meant to *illustrate* the effect of class imbalance:
    it usually predicts only the majority class, leading to apparently
    reasonable accuracy but zero recall and F1 for the minority class, as
    discussed in the PDF for imbalanced problems. 
    """
    confusion_terms: Dict[str, int]
    metrics: Dict[str, float]

    def as_dict(self) -> Dict[str, Dict[str, float]]:
        return {
            "confusion_terms": self.confusion_terms,
            "metrics": self.metrics,
        }


def run_dummy_most_frequent_baseline(
    df: pd.DataFrame,
    *,
    feature_cols: Sequence[str],
    target_col: str = "y",
    test_size: float = 0.2,
    random_state: int = 42,
) -> DummyBaselineResult:
    """Run a 'most_frequent' DummyClassifier to illustrate imbalance.

    Parameters
    ----------
    df:
        DataFrame with features and binary target.
    feature_cols:
        Columns used as input X (raw; preprocessing comes later).
    target_col:
        Name of the binary target column (e.g., 'readmitted_30d' or 'y').
    test_size:
        Fraction reserved for the illustrative test split.
    random_state:
        Seed for the train/test partition (kept fixed for reproducibility).

    Returns
    -------
    DummyBaselineResult
        Confusion-matrix terms TN/FP/FN/TP and metrics:
        accuracy, precision, recall, F1.
    """
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found in df.")

    X = df[feature_cols]
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state,
    )

    clf = DummyClassifier(strategy="most_frequent")
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    tn, fp, fn, tp = metrics.confusion_matrix(y_test, y_pred).ravel()

    metrics_dict = {
        "accuracy": float(metrics.accuracy_score(y_test, y_pred)),
        "precision": float(metrics.precision_score(y_test, y_pred, zero_division=0)),
        "recall": float(metrics.recall_score(y_test, y_pred, zero_division=0)),
        "f1": float(metrics.f1_score(y_test, y_pred, zero_division=0)),
    }

    confusion_terms = {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

    return DummyBaselineResult(confusion_terms=confusion_terms, metrics=metrics_dict)


Writing section4_eda_imbalance.py


In [7]:
%%writefile test_section4_eda_imbalance.py
import unittest
import numpy as np
import pandas as pd

import section4_eda_imbalance as s4


class TestSection4EDAImbalance(unittest.TestCase):
    def setUp(self):
        # Pequeño dataframe con números, categóricas y target binario
        self.df = pd.DataFrame(
            {
                "feat1": [1, 2, 3, 4],
                "feat2": ["a", "?", "", "b"],       # '?' y '' se consideran missing
                "feat3": [np.nan, "x", "y", "z"],  # NaN se considera missing
                "y": [0, 0, 1, 0],
            }
        )

    # ---- 4.1: class imbalance -----------------------------------------

    def test_compute_class_imbalance_basic(self):
        summary = s4.compute_class_imbalance(self.df["y"])
        self.assertEqual(summary.N, 4)
        self.assertEqual(summary.n_pos, 1)
        self.assertEqual(summary.n_neg, 3)
        self.assertAlmostEqual(summary.pos_rate, 0.25)
        self.assertAlmostEqual(summary.neg_rate, 0.75)
        d = summary.as_dict()
        self.assertIn("n_pos", d)
        self.assertIn("pos_rate", d)

    # ---- 4.2: missingness predicates ----------------------------------

    def test_is_missing_handles_nan_and_tokens(self):
        self.assertTrue(s4.is_missing(None))
        self.assertTrue(s4.is_missing(np.nan))
        self.assertTrue(s4.is_missing("   "))          # vacío tras strip
        self.assertTrue(s4.is_missing("?"))
        self.assertTrue(s4.is_missing(" unknown "))    # token normalizado
        self.assertFalse(s4.is_missing("value"))

    def test_missingness_by_feature_rates_and_counts(self):
        mf = s4.missingness_by_feature(
            self.df,
            feature_cols=["feat1", "feat2", "feat3"],
        )
        # feat1 no tiene missing
        self.assertEqual(int(mf.loc["feat1", "missing_count"]), 0)
        self.assertAlmostEqual(float(mf.loc["feat1", "missing_rate"]), 0.0)

        # feat2: '?' y '' -> 2 de 4 => 0.5
        self.assertEqual(int(mf.loc["feat2", "missing_count"]), 2)
        self.assertAlmostEqual(float(mf.loc["feat2", "missing_rate"]), 0.5)

        # feat3: un NaN -> 1 de 4 => 0.25
        self.assertEqual(int(mf.loc["feat3", "missing_count"]), 1)
        self.assertAlmostEqual(float(mf.loc["feat3", "missing_rate"]), 0.25)

    def test_missingness_by_row_summary_and_high_missing_fraction(self):
        res = s4.missingness_by_row(
            self.df,
            feature_cols=["feat1", "feat2", "feat3"],
            high_missing_threshold=0.30,
        )

        # Comprobamos longitudes
        self.assertEqual(len(res.missing_per_row), 4)
        self.assertEqual(len(res.missing_rate_per_row), 4)

        # Cuentas esperadas por fila:
        # fila0: NaN en feat3 -> 1
        # fila1: '?' en feat2 -> 1
        # fila2: '' en feat2 -> 1
        # fila3: sin missing -> 0
        self.assertListEqual(res.missing_per_row.tolist(), [1, 1, 1, 0])

        # 3 de 4 filas con ratio >= 1/3 (~0.33) > 0.30
        self.assertAlmostEqual(res.high_missing_fraction, 0.75)

        d = res.as_dict()
        self.assertIn("missing_per_row_summary", d)
        self.assertIn("high_missing_fraction", d)

    # ---- 4.3: top-k tables for high-cardinality categoricals ----------

    def test_topk_table_collapses_missing_and_computes_coverage(self):
        top, nunique, coverage = s4.topk_table(self.df["feat2"], k=2)
        # categorías: 'a', 'b', '__MISSING__'
        self.assertEqual(nunique, 3)
        # top-2 cubre '__MISSING__'(2) + 'a'(1) = 3/4 => 0.75
        self.assertAlmostEqual(coverage, 0.75)

        # la categoría '__MISSING__' debe estar en la tabla
        self.assertIn("__MISSING__",
                      top.index.astype(str).tolist())

    # ---- 4.4: dummy baseline under imbalance --------------------------

    def test_run_dummy_most_frequent_baseline_behaviour(self):
        # Dataset más grande e imbalance más claro
        df_large = pd.DataFrame(
            {
                "f1": np.arange(50),
                "y": [0] * 40 + [1] * 10,  # 80% negativos
            }
        )

        result = s4.run_dummy_most_frequent_baseline(
            df_large,
            feature_cols=["f1"],
            target_col="y",
            test_size=0.2,
            random_state=0,
        )

        cm = result.confusion_terms
        metrics_dict = result.metrics

        # Estrategia 'most_frequent' debe predecir solo la clase mayoritaria (0)
        self.assertEqual(cm["tp"], 0)
        self.assertEqual(cm["fp"], 0)
        self.assertGreater(cm["tn"], 0)
        self.assertGreater(cm["fn"], 0)

        # Bajo desequilibrio: accuracy > 0, pero recall y F1 del positivo = 0
        self.assertGreater(metrics_dict["accuracy"], 0.0)
        self.assertAlmostEqual(metrics_dict["recall"], 0.0)
        self.assertAlmostEqual(metrics_dict["f1"], 0.0)

        d = result.as_dict()
        self.assertIn("confusion_terms", d)
        self.assertIn("metrics", d)


if __name__ == "__main__":
    unittest.main(verbosity=2)


Writing test_section4_eda_imbalance.py


In [8]:
!python -m unittest -v test_section4_eda_imbalance.py


test_compute_class_imbalance_basic (test_section4_eda_imbalance.TestSection4EDAImbalance.test_compute_class_imbalance_basic) ... ok
test_is_missing_handles_nan_and_tokens (test_section4_eda_imbalance.TestSection4EDAImbalance.test_is_missing_handles_nan_and_tokens) ... ok
test_missingness_by_feature_rates_and_counts (test_section4_eda_imbalance.TestSection4EDAImbalance.test_missingness_by_feature_rates_and_counts) ... ok
test_missingness_by_row_summary_and_high_missing_fraction (test_section4_eda_imbalance.TestSection4EDAImbalance.test_missingness_by_row_summary_and_high_missing_fraction) ... c:\Users\Cris-SX\Desktop\UA\CHAD-MASTER-IA\TAU\tau-final-project\code\lvl1\lvl2\tau-final-code\section4_eda_imbalance.py:217: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  miss_matrix = df[feature_cols].applymap(
ok
test_run_dummy_most_frequent_baseline_behaviour (test_section4_eda_imbalance.TestSection4EDAImbalance.test_run_dummy_most_frequent_baseline_behavio

In [ ]:
%%writefile section5_preprocessing_pipeline.py

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

import math

# ---------------------------------------------------------------------
# Section 5 (PDF-aligned):
# - Numeric values: scaling/normalization (StandardScaler / MinMaxScaler)
# - Categorical values: binarization (dummies / one-hot)
# - Missing/unknown: remove variable/sample or impute (Simple/KNN/Iterative)
# - No leakage: all learned steps must be fit inside CV folds via Pipeline
# ---------------------------------------------------------------------

DEFAULT_MISSING_TOKENS: Tuple[str, ...] = (
    "",
    "?",
    "NA",
    "N/A",
    "NONE",
    "NULL",
    "NAN",
    "UNKNOWN",
    "UNKNOWN/INVALID",
)

DEFAULT_ID_COLS: Tuple[str, ...] = ("encounter_id", "patient_nbr")
DEFAULT_TARGET_COLS: Tuple[str, ...] = ("readmitted", "readmitted_30d", "readmitted_3class", "y")

# Ordinal default: only AGE is treated as ordinal (safe + clinically meaningful order).
AGE_ORDER: Tuple[str, ...] = (
    "[0-10)",
    "[10-20)",
    "[20-30)",
    "[30-40)",
    "[40-50)",
    "[50-60)",
    "[60-70)",
    "[70-80)",
    "[80-90)",
    "[90-100)",
)

# Integer-coded fields that are nominal categories (avoid imposing fake geometry)
DEFAULT_FORCE_CATEGORICAL: Tuple[str, ...] = (
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
)

# Count-like numeric fields (plus weight if present)
DEFAULT_NUMERIC_HINTS: Tuple[str, ...] = (
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "weight",
)


def _normalize_token(x: str) -> str:
    return x.strip().upper()


def _missing_token_set(tokens: Sequence[str]) -> set:
    return {_normalize_token(t) for t in tokens}

class MissingTokenCleaner(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        missing_tokens=("?", "UNKNOWN/INVALID", ""),
        *,
        case_insensitive: bool = True,
        strip: bool = True,
        treat_empty_as_missing: bool = True,
    ):
        self.missing_tokens = tuple(missing_tokens)
        self.case_insensitive = case_insensitive
        self.strip = strip
        self.treat_empty_as_missing = treat_empty_as_missing

    def fit(self, X, y=None):
        toks = []
        for t in self.missing_tokens:
            s = str(t)
            s = s.strip() if self.strip else s
            s = s.upper() if self.case_insensitive else s
            toks.append(s)
        self._missing_set_ = set(toks)
        return self

    def transform(self, X):
        # Soporta DataFrame y arrays
        if isinstance(X, pd.DataFrame):
            out = X.copy()
            missing_set = getattr(self, "_missing_set_", None)
            if missing_set is None:
                self.fit(X)
                missing_set = self._missing_set_

            for c in out.columns:
                s = out[c]
                if not (pd.api.types.is_object_dtype(s.dtype) or pd.api.types.is_string_dtype(s.dtype)):
                    continue

                def _clean(v):
                    if v is None:
                        return np.nan
                    try:
                        if pd.isna(v):
                            return np.nan
                    except Exception:
                        pass

                    if isinstance(v, str):
                        vv = v.strip() if self.strip else v
                        if self.treat_empty_as_missing and vv == "":
                            return np.nan
                        key = vv.upper() if self.case_insensitive else vv
                        return np.nan if key in missing_set else vv

                    vv = str(v)
                    vv = vv.strip() if self.strip else vv
                    key = vv.upper() if self.case_insensitive else vv
                    return np.nan if key in missing_set else v

                out[c] = s.map(_clean)

            return out

        # array-like (n_samples, n_features)
        arr = np.asarray(X, dtype=object)
        missing_set = getattr(self, "_missing_set_", None)
        if missing_set is None:
            # fit “perezoso”
            self.fit(pd.DataFrame(arr))
            missing_set = self._missing_set_

        def _clean_scalar(v):
            if v is None:
                return np.nan
            try:
                if pd.isna(v):
                    return np.nan
            except Exception:
                pass
            if isinstance(v, str):
                vv = v.strip() if self.strip else v
                if self.treat_empty_as_missing and vv == "":
                    return np.nan
                key = vv.upper() if self.case_insensitive else vv
                return np.nan if key in missing_set else vv
            vv = str(v)
            vv = vv.strip() if self.strip else vv
            key = vv.upper() if self.case_insensitive else vv
            return np.nan if key in missing_set else v

        vfunc = np.vectorize(_clean_scalar, otypes=[object])
        return vfunc(arr)



class NumericCoercer(BaseEstimator, TransformerMixin):
    """Coerce selected columns to numeric (float), invalid parsing -> NaN.
    """

    def __init__(self, numeric_cols: Sequence[str]):
        self.numeric_cols = numeric_cols  # DO NOT copy/convert (clone compatibility)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            return X
        df = X.copy()
        for col in self.numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")
        return df

class CategoricalCaster(BaseEstimator, TransformerMixin):
    """
    Cast categorical values to strings (keeping missing as np.nan).
    Prevents mixed types (e.g., int-coded categories + string missing filler).
    """

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_arr = self._to_2d_array(X).astype("object", copy=True)
        out = np.empty_like(X_arr, dtype="object")

        for i in range(X_arr.shape[0]):
            for j in range(X_arr.shape[1]):
                v = X_arr[i, j]
                if v is None:
                    out[i, j] = np.nan
                    continue
                try:
                    if pd.isna(v):
                        out[i, j] = np.nan
                        continue
                except Exception:
                    pass
                out[i, j] = str(v)

        return out

    @staticmethod
    def _to_2d_array(X):
        if isinstance(X, pd.DataFrame):
            return X.to_numpy(dtype="object")
        if isinstance(X, pd.Series):
            return X.to_frame().to_numpy(dtype="object")
        X_arr = np.asarray(X, dtype="object")
        if X_arr.ndim == 1:
            X_arr = X_arr.reshape(-1, 1)
        return X_arr
from sklearn.base import BaseEstimator, TransformerMixin

class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        min_count: int = 10,
        *,
        min_frequency: float | None = None,
        other_label: str = "__OTHER__",
        missing_label: str = "__MISSING__",
    ):
        self.min_count = int(min_count)
        self.min_frequency = min_frequency
        self.other_label = other_label
        self.missing_label = missing_label

    def fit(self, X, y=None):
        arr = np.asarray(X, dtype=object)
        if arr.ndim == 1:
            arr = arr.reshape(-1, 1)

        n_rows = arr.shape[0]

        # If min_frequency is provided, convert it to an absolute threshold per fit()
        freq_count = 0
        if self.min_frequency is not None:
            if not (0.0 < float(self.min_frequency) <= 1.0):
                raise ValueError(f"min_frequency must be in (0,1], got {self.min_frequency}")
            freq_count = int(math.ceil(float(self.min_frequency) * n_rows))

        effective_min = max(1, self.min_count, freq_count)

        allowed = []
        for j in range(arr.shape[1]):
            col = arr[:, j]

            col2 = np.array(
                [
                    self.missing_label
                    if (v is None or pd.isna(v))
                    else v
                    for v in col
                ],
                dtype=object,
            )

            vc = pd.Series(col2).value_counts(dropna=False)
            keep = set(vc[vc >= effective_min].index.tolist())

            # Always allow missing bucket
            keep.add(self.missing_label)

            allowed.append(keep)

        self.allowed_categories_ = allowed
        self.effective_min_count_ = effective_min
        return self

    def transform(self, X):
        arr = np.asarray(X, dtype=object)
        if arr.ndim == 1:
            arr = arr.reshape(-1, 1)

        if not hasattr(self, "allowed_categories_"):
            self.fit(arr)

        out = arr.copy()
        for j in range(out.shape[1]):
            keep = self.allowed_categories_[j]

            def _map(v):
                if v is None or pd.isna(v):
                    return self.missing_label
                if v == self.missing_label:
                    return self.missing_label
                return v if v in keep else self.other_label

            out[:, j] = np.vectorize(_map, otypes=[object])(out[:, j])

        return out

class ToNumeric(BaseEstimator, TransformerMixin):
    """
    Convert input columns to numeric (float) using pandas.to_numeric(errors='coerce').
    This is mandatory for numeric-like string columns (e.g., 'weight' = "70") so that
    median imputation + scaling work reliably.
    """

    def __init__(self, dtype: str = "float64"):
        self.dtype = dtype

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # DataFrame path (best case)
        if isinstance(X, pd.DataFrame):
            return X.apply(pd.to_numeric, errors="coerce").to_numpy(dtype=self.dtype)

        # numpy / array-like path
        arr = np.asarray(X, dtype=object)
        if arr.ndim == 1:
            arr = arr.reshape(-1, 1)

        out = np.empty(arr.shape, dtype=self.dtype)
        for j in range(arr.shape[1]):
            out[:, j] = pd.to_numeric(pd.Series(arr[:, j]), errors="coerce").to_numpy(dtype=self.dtype)
        return out

def make_onehot_encoder(*, sparse_output: bool):
    # sklearn >= 1.2 uses sparse_output; older uses sparse
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=sparse_output)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=sparse_output)

def _make_one_hot_encoder(*, sparse: bool, handle_unknown: str = "ignore") -> OneHotEncoder:
    # sklearn compatibility (sparse vs sparse_output)
    try:
        return OneHotEncoder(handle_unknown=handle_unknown, sparse_output=sparse)
    except TypeError:
        return OneHotEncoder(handle_unknown=handle_unknown, sparse=sparse)


def _make_ordinal_encoder(categories: List[List[str]]) -> OrdinalEncoder:
    # sklearn compatibility: handle_unknown might not exist in older versions
    try:
        return OrdinalEncoder(
            categories=categories,
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        )
    except TypeError:
        return OrdinalEncoder(categories=categories)


@dataclass(frozen=True)
class DiabetesPreprocessConfig:
    # Missingness tokens and core columns
    missing_tokens: Tuple[str, ...] = DEFAULT_MISSING_TOKENS
    id_cols: Tuple[str, ...] = DEFAULT_ID_COLS
    target_cols: Tuple[str, ...] = DEFAULT_TARGET_COLS

    # Ordinal specs
    ordinal_cols: Mapping[str, Sequence[str]] = field(default_factory=lambda: {"age": AGE_ORDER})

    # Force integer-coded categoricals
    force_categorical_cols: Tuple[str, ...] = DEFAULT_FORCE_CATEGORICAL

    # Numeric imputation + scaling
    numeric_impute_strategy: str = "median"   # PDF: mean/median are standard
    scale_numeric: bool = True               # PDF: scaling/normalization recommended for numeric for many models
    scaler: str = "standard"                 # 'standard' or 'minmax' (PDF mentions both)

    # Categorical handling
    cat_impute_strategy: str = "constant"    # constant -> explicit "__MISSING__" bucket
    cat_impute_fill_value: str = "__MISSING__"
    rare_min_count: int = 50
    rare_min_frequency: Optional[float] = None
    rare_other_label: str = "__OTHER__"

    # Output format
    onehot_sparse: bool = True


@dataclass(frozen=True)
class FeatureGroups:
    numeric: List[str]
    ordinal: List[str]
    categorical: List[str]
    dropped: List[str]


def infer_feature_groups(df: pd.DataFrame, config: DiabetesPreprocessConfig) -> FeatureGroups:
    cols = list(df.columns)

    id_set = set(config.id_cols)
    target_set = set(config.target_cols)

    dropped = [c for c in cols if c in id_set]
    usable = [c for c in cols if c not in id_set and c not in target_set]

    ordinal = [c for c in usable if c in set(config.ordinal_cols.keys())]
    remaining = [c for c in usable if c not in set(ordinal)]

    force_cat = set(config.force_categorical_cols)

    # Numeric:
    # (1) known count-like fields present
    # (2) detected numeric dtypes, excluding forced categoricals
    numeric: List[str] = [c for c in remaining if c in set(DEFAULT_NUMERIC_HINTS)]
    numeric_set = set(numeric)

    for c in remaining:
        if c in numeric_set or c in force_cat:
            continue
        if pd.api.types.is_numeric_dtype(df[c].dtype) and not pd.api.types.is_bool_dtype(df[c].dtype):
            numeric.append(c)
            numeric_set.add(c)

    # Everything else -> categorical (includes forced categoricals)
    categorical = [c for c in remaining if c not in numeric_set]
    return FeatureGroups(
        numeric=list(numeric),
        ordinal=list(ordinal),
        categorical=list(categorical),
        dropped=list(dropped),
    )


def build_preprocessor(df: pd.DataFrame, config: Optional[DiabetesPreprocessConfig] = None) -> Pipeline:
    """
    Single-Source-of-Truth preprocessing pipeline.

    Guarantees:
    - Unified missingness: tokens -> NaN
    - Categorical: one-hot (handle_unknown='ignore') + optional rare-category grouping
    - Numeric: median imputation + (optional) scaling/normalization
    - Ordinal: explicit ordered encoding (age)
    - Leakage-safe when used inside CV: all learned steps are fit on training folds only.
    """
    if config is None:
        config = DiabetesPreprocessConfig()

    groups = infer_feature_groups(df, config)

    # Numeric pipeline
    num_steps = [
        ("to_numeric", ToNumeric()),  # <<< AQUÍ VA
        ("imputer", SimpleImputer(strategy=config.numeric_impute_strategy)),
    ]

    if config.scale_numeric:
        scaler_name = config.scaler.lower()
        if scaler_name == "standard":
            num_steps.append(("scaler", StandardScaler()))
        elif scaler_name == "minmax":
            num_steps.append(("scaler", MinMaxScaler()))
        else:
            raise ValueError(f"Unknown scaler: {config.scaler}. Use 'standard' or 'minmax'.")

    num_pipe = Pipeline(steps=num_steps)

    # Ordinal pipeline (age)
    if groups.ordinal:
        categories: List[List[str]] = [list(config.ordinal_cols[c]) for c in groups.ordinal]
        ord_pipe = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="constant", fill_value=config.cat_impute_fill_value)),
                ("ordinal", _make_ordinal_encoder(categories=categories)),
            ]
        )
    else:
        ord_pipe = "drop"

    # Categorical pipeline
    if config.cat_impute_strategy == "constant":
        cat_imputer = SimpleImputer(strategy="constant", fill_value=config.cat_impute_fill_value)
    elif config.cat_impute_strategy == "most_frequent":
        cat_imputer = SimpleImputer(strategy="most_frequent")
    else:
        raise ValueError("cat_impute_strategy must be 'constant' or 'most_frequent'.")

    onehot = make_onehot_encoder(sparse_output=config.onehot_sparse)

    cat_pipe = Pipeline(
        steps=[
            ("cast_to_str", CategoricalCaster()),
            ("imputer", cat_imputer),
            ("rare", RareCategoryGrouper(
                min_count=config.rare_min_count,
                min_frequency=config.rare_min_frequency,
                other_label=config.rare_other_label,
                missing_label=config.cat_impute_fill_value,
            )),
            ("onehot", onehot),
        ]
    )

    transformers = []
    if groups.numeric:
        transformers.append(("num", num_pipe, groups.numeric))
    if groups.ordinal:
        transformers.append(("ord", ord_pipe, groups.ordinal))
    if groups.categorical:
        transformers.append(("cat", cat_pipe, groups.categorical))

    ct = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    sparse_threshold=0.0 if not config.onehot_sparse else 0.3,
)

    return Pipeline(
        steps=[
            ("clean_missing_tokens", MissingTokenCleaner(missing_tokens=config.missing_tokens)),
            ("coerce_numeric", NumericCoercer(numeric_cols=groups.numeric)),
            ("features", ct),
        ]
    )


Overwriting section5_preprocessing_pipeline.py


In [7]:
%%writefile test_section5_preprocessing_pipeline.py

import unittest

import math


import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin


import section5_preprocessing_pipeline as s5


class TestSection5PreprocessingPipeline(unittest.TestCase):
    def _toy_df(self):
        # Mini dataset that matches the real schema patterns:
        # - ID columns to drop
        # - ordinal age
        # - numeric counts + numeric-like strings (weight)
        # - int-coded categorical (admission_type_id)
        # - missing tokens ("?", "Unknown/Invalid")
        # - rare categorical values
        df = pd.DataFrame(
            {
                "encounter_id": [1, 2, 3, 4, 5, 6],
                "patient_nbr": [10, 11, 12, 13, 14, 15],
                "age": ["[0-10)", "[10-20)", "?", "[30-40)", "[40-50)", "[50-60)"],
                "weight": ["?", "80", "90", "?", "Unknown/Invalid", "70"],
                "num_lab_procedures": [41, 59, 11, 44, np.nan, 50],
                "admission_type_id": [6, 1, 1, 1, 6, 6],
                "medical_specialty": ["Pediatrics-Endocrinology", "?", "Cardiology", "Cardiology", "RareSpec", "Cardiology"],
                "diag_1": ["250.83", "276", "648", "8", "777", "8"],
                "readmitted_30d": [0, 1, 0, 0, 1, 0],
            }
        )
        return df

    def test_missing_token_cleaner_replaces_tokens_with_nan(self):
        df = self._toy_df()
        cleaner = s5.MissingTokenCleaner(missing_tokens=("?", "UNKNOWN/INVALID"))
        out = cleaner.transform(df[["medical_specialty", "weight", "age"]])

        # '?' and 'Unknown/Invalid' should become NaN
        self.assertTrue(pd.isna(out.loc[1, "medical_specialty"]))
        self.assertTrue(pd.isna(out.loc[0, "weight"]))
        self.assertTrue(pd.isna(out.loc[4, "weight"]))
        self.assertTrue(pd.isna(out.loc[2, "age"]))

    def test_infer_feature_groups_expected(self):
        df = self._toy_df()
        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=False, rare_min_count=2)

        groups = s5.infer_feature_groups(df, cfg)

        # IDs dropped
        self.assertIn("encounter_id", groups.dropped)
        self.assertIn("patient_nbr", groups.dropped)

        # age is ordinal
        self.assertIn("age", groups.ordinal)

        # numeric includes weight + num_lab_procedures
        self.assertIn("weight", groups.numeric)
        self.assertIn("num_lab_procedures", groups.numeric)

        # admission_type_id forced categorical (not numeric)
        self.assertIn("admission_type_id", groups.categorical)
        self.assertNotIn("admission_type_id", groups.numeric)

    def test_preprocessor_fit_transform_no_crash_and_no_nan(self):
        df = self._toy_df()
        X = df.drop(columns=["readmitted_30d"])
        y = df["readmitted_30d"]

        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=False, rare_min_count=2)
        pre = s5.build_preprocessor(df, cfg)

        pre.fit(X, y)
        Xt = pre.transform(X)

        self.assertEqual(Xt.shape[0], len(df))
        # After imputation/encoding there should be no NaNs (Ordinal unknown becomes -1, not NaN)
        self.assertFalse(np.isnan(np.asarray(Xt)).any())

    def test_unseen_category_does_not_break_transform(self):
        df = self._toy_df()
        train = df.iloc[:4].copy()
        test = df.iloc[4:].copy()

        X_train = train.drop(columns=["readmitted_30d"])
        y_train = train["readmitted_30d"]
        X_test = test.drop(columns=["readmitted_30d"])

        # Force rare grouping to be aggressive, so unseen/rare go to __OTHER__
        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=False, rare_min_count=2)
        pre = s5.build_preprocessor(train, cfg)

        pre.fit(X_train, y_train)
        _ = pre.transform(X_test)  # should not raise

        self.assertTrue(True)

    def test_rare_category_grouper_maps_rare_to_other(self):
        X = np.array(
            [
                ["A"],
                ["A"],
                ["B"],  # rare if min_count=2
                ["__MISSING__"],
            ],
            dtype=object,
        )
        g = s5.RareCategoryGrouper(min_count=2, other_label="__OTHER__", missing_label="__MISSING__")
        g.fit(X)
        Xt = g.transform(np.array([["B"], ["A"], ["C"], [np.nan]], dtype=object))

        # B is rare -> __OTHER__; C unseen -> __OTHER__; nan -> __MISSING__
        self.assertEqual(Xt[0, 0], "__OTHER__")
        self.assertEqual(Xt[1, 0], "A")
        self.assertEqual(Xt[2, 0], "__OTHER__")
        self.assertEqual(Xt[3, 0], "__MISSING__")

    def test_pipeline_works_inside_cross_validation(self):
        df = self._toy_df()
        X = df.drop(columns=["readmitted_30d"])
        y = df["readmitted_30d"]

        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=False, rare_min_count=2)
        pre = s5.build_preprocessor(df, cfg)

        pipe = Pipeline(
            steps=[
                ("pre", pre),
                ("clf", LogisticRegression(max_iter=1000)),
            ]
        )

        # ROC-AUC requires BOTH classes in every test fold.
        # With tiny toy data, choose n_splits <= min(class_count) to avoid undefined AUC.
        counts = pd.Series(y).value_counts()
        min_class = int(counts.min())
        if min_class < 2:
            self.skipTest("Need at least 2 samples per class to compute ROC-AUC in CV.")

        from sklearn.model_selection import StratifiedKFold

        n_splits = min(3, min_class)  # ensures each fold has at least 1 sample of each class
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)

        scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
        self.assertEqual(len(scores), n_splits)
        self.assertTrue(np.isfinite(scores).all())



Overwriting test_section5_preprocessing_pipeline.py


In [8]:
!python -m unittest -v test_section5_preprocessing_pipeline.py 


test_infer_feature_groups_expected (test_section5_preprocessing_pipeline.TestSection5PreprocessingPipeline.test_infer_feature_groups_expected) ... ok
test_missing_token_cleaner_replaces_tokens_with_nan (test_section5_preprocessing_pipeline.TestSection5PreprocessingPipeline.test_missing_token_cleaner_replaces_tokens_with_nan) ... ok
test_pipeline_works_inside_cross_validation (test_section5_preprocessing_pipeline.TestSection5PreprocessingPipeline.test_pipeline_works_inside_cross_validation) ... ERROR
test_preprocessor_fit_transform_no_crash_and_no_nan (test_section5_preprocessing_pipeline.TestSection5PreprocessingPipeline.test_preprocessor_fit_transform_no_crash_and_no_nan) ... ok
test_rare_category_grouper_maps_rare_to_other (test_section5_preprocessing_pipeline.TestSection5PreprocessingPipeline.test_rare_category_grouper_maps_rare_to_other) ... ok
test_unseen_category_does_not_break_transform (test_section5_preprocessing_pipeline.TestSection5PreprocessingPipeline.test_unseen_category_